# 04 · Run a language model on the task set

**Why.** The benchmark exists to score language-model agents. This notebook lets one model play every task of the
first task set several times and saves every game, so that notebook 05 can score them against the oracle, the
human games and a random player.

**What.** The model plays through an OpenAI-compatible chat API (see `wikigame/agent.py`): the whole game so far stays in
its context, one observation per turn. Settings:

* **Repeats.** Each task is played 3 times with sampling (temperature 0.7, a different seed per run), because one
  game per task would mix the model's skill with luck.
* **Step budget.** 2·V* + 10 actions, where V* is the fewest actions the task needs. That leaves room for a
  detour while still ending games that wander. The step budget is an open design choice of the benchmark.
* **Reasoning effort.** Read from the environment variable `REASONING_EFFORT` (`none`, `low`, `medium` or `xhigh`
  for Qwen3.8), so one run of the notebook covers one level.

Games are appended to `results/<model>/effort-<level>.jsonl` as soon as they end. A rerun skips the games already
saved, so an interrupted run can simply be started again.

**Environment.** `LLM_BASE_URL` and `LLM_API_KEY` point to the chat API. `TASK_LIMIT` (optional) plays only the
first tasks, for a smoke test, and `RESULTS_DIR` overrides where games are saved.

**Runtime.** It depends on the model and the reasoning level. The runtime of each run is printed at the end
and summarised in notebook 05.

In [ ]:
import json, os, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from wikigame.world import World, ROOT
from wikigame.metrics import Graphs, Oracle
from wikigame.env import WikiGameEnv
from wikigame.agent import play

MODEL = "llm-balanced"            # gateway alias; the gateway serves it with qwen3.8:27b
MODEL_NAME = "qwen3.8-27b"        # used for the results folder
EFFORT = os.environ.get("REASONING_EFFORT", "none")
TEMPERATURE, RUNS, PARALLEL = 0.7, 3, 8
budget = lambda V_star: 2 * V_star + 10

out_dir = Path(os.environ.get("RESULTS_DIR", ROOT / "results" / MODEL_NAME))
out_dir.mkdir(parents=True, exist_ok=True)
OUT = out_dir / f"effort-{EFFORT}.jsonl"

world = World.load("wikispeedia")
g = Graphs.build(world)
tasks = json.load(open(world.dir / "tasks" / "index.json"))["tasks"]
if os.environ.get("TASK_LIMIT"):
    tasks = tasks[: int(os.environ["TASK_LIMIT"])]
oracles = {t["target"]: Oracle(world, g, t["target"]) for t in tasks}
print(f"{len(tasks)} tasks, reasoning effort {EFFORT!r}, writing to {OUT.relative_to(ROOT) if OUT.is_relative_to(ROOT) else OUT}")

In [ ]:
done = set()
if OUT.exists():
    done = {(r["task"], r["run"]) for r in map(json.loads, OUT.read_text().splitlines())}
# all tasks of run 0 first, then run 1, ...: a partial file still covers every task evenly
jobs = [(run, t) for run in range(RUNS) for t in tasks if (t["id"], run) not in done]
print(f"{len(done)} games already saved, {len(jobs)} to play")

lock = threading.Lock()
def one_game(run, t):
    env = WikiGameEnv(world, oracles[t["target"]], t["start"], max_steps=budget(t["V_star"]))
    game = play(env, MODEL, temperature=TEMPERATURE, max_tokens=16384, reasoning_effort=EFFORT, seed=run)
    row = {"task": t["id"], "run": run, "model": MODEL, "model_name": MODEL_NAME, "reasoning_effort": EFFORT,
           "temperature": TEMPERATURE, "max_steps": budget(t["V_star"]), "finished_at": time.strftime("%Y-%m-%dT%H:%M:%S"), **game}
    with lock, open(OUT, "a") as f:
        f.write(json.dumps(row) + "\n")
    return row

t0 = time.time()
with ThreadPoolExecutor(PARALLEL) as pool:
    futures = [pool.submit(one_game, run, t) for run, t in jobs]
    for i, fut in enumerate(as_completed(futures), 1):
        try:
            r = fut.result()
            s = r["score"]["actions"]
            print(f"[{i}/{len(jobs)}] {r['task']} run {r['run']}: {'reached' if s['success'] else 'did not reach'} in {s['steps']} steps "
                  f"(optimum {s['optimal_steps']}), {r['seconds']:.0f} s", flush=True)
        except Exception as error:   # a game that fails (e.g. the API is down) is left for the next run
            print(f"[{i}/{len(jobs)}] failed: {error!r}", flush=True)
print(f"run time {(time.time() - t0) / 60:.1f} min")